# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes -- signal checks first

**Plain-words rule:** *A page is a CTR-fix candidate if it already ranks well (page 1 or
striking distance) with enough traffic to trust its numbers, but its click-through rate sits
below what similar pages at the same position tier get. That gap means the listing itself
(title/meta) is probably costing clicks -- not the ranking.*

Before coding that, I checked the two signals it leans on. Both come from the starter dataset's
current-snapshot columns; volume filter (`impressions_90d >= 100`) applied throughout so CTR
numbers aren't noise from a handful of impressions.

**Signal 1 (flag-linked -- mirrors FlyRank's `needs_ctr_fix` logic): does CTR actually vary by
position tier?** If it doesn't, "compare CTR within your tier" is meaningless.

**Signal 2 (flag-linked -- mirrors FlyRank's refresh flags): does staleness predict CTR?** This
is the one I expected to lean on too, going in.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df = df[df["avg_position"] > 0].copy()          # 0 means "no data", not rank zero
vol = df[df["impressions_90d"] >= 100].copy()   # volume floor so CTR isn't noise

print(f"Rows after position + volume filters: {len(vol):,} of {len(df):,}")
print()

print("=== Signal 1: CTR by position tier (n printed) ===")
sig1 = vol.groupby("position_tier")["ctr"].agg(["count", "median"]).sort_values("median", ascending=False)
print(sig1)
print()

print("=== Signal 2: CTR by freshness tier (n printed) ===")
sig2 = vol.groupby("freshness_tier")["ctr"].agg(["count", "median"])
print(sig2)


Rows after position + volume filters: 22,006 of 28,795

=== Signal 1: CTR by position tier (n printed) ===
               count  median
position_tier               
page_1          8633    0.23
top_3            533    0.19
striking        5903    0.15
page_3_5        6058    0.06
deep             879    0.00

=== Signal 2: CTR by freshness tier (n printed) ===
                count  median
freshness_tier               
0-30            13735   0.150
181+               35   0.180
31-90             152   0.055
91-180           8084   0.130


**Signal 1 verdict: MIXED.** The coarse pattern is real -- page-1-or-better tiers (`top_3`
0.19, `page_1` 0.23, `striking` 0.15) sit far above `page_3_5` (0.06) and `deep` (0.00). But it's
not cleanly monotonic: `page_1` actually edges out `top_3`. So position tier is a real, usable
signal for "what CTR should this page get" -- but only as a *group* comparison, not a strict
"better rank = strictly better CTR" rule. That's exactly why the score below compares each page
to its own tier's median rather than assuming a smooth curve.

**Signal 2 verdict: MIXED, leaning toward NOT USABLE.** Median CTR by freshness tier doesn't
show staleness hurting CTR in the two buckets that hold 99% of the data (`0-30` days: 0.150 CTR,
n=13,735; `91-180` days: 0.130 CTR, n=8,084 -- barely different, and not in the direction
"staler is worse" would predict cleanly). The two extreme buckets (`31-90`: 0.055, n=152;
`181+`: 0.180, n=35) are too small to trust and don't even agree with each other on direction.

**This negative is the win the skill promised:** I went in planning to use staleness as a
multiplier in the score. It doesn't hold up, so I'm dropping it from the rule entirely rather
than forcing it in -- keeping `freshness_tier` only as a column in the output for a human to
glance at, never as something the score depends on.


## 2. Build the ranked queue (writes the CSV)

**Final rule, using only the signal that survived:**
`score = (tier_median_ctr - ctr) * impressions_90d`, computed only for pages that are already
well-ranked and have enough volume to trust the CTR reading. Bigger gap x more traffic = bigger
opportunity. Transparent, no fitted weights, readable in one line -- per the baseline pattern.

- **Eligibility gate:** `position_tier` in `{top_3, page_1, striking}` AND `impressions_90d >= 100`
- **Score:** `(tier_median_ctr - ctr) * impressions_90d`, only for pages below their tier median
  (score = 0 / excluded otherwise)
- **Reason code (one, applied to every flagged row):** `position_ok_ctr_below_tier_peers`
- **Action (one, applied to every flagged row):** `review_listing_ctr`


In [2]:
import os

eligible = vol[vol["position_tier"].isin(["top_3", "page_1", "striking"])].copy()

tier_median = eligible.groupby("position_tier")["ctr"].transform("median")
eligible["tier_median_ctr"] = tier_median
eligible["ctr_gap"] = eligible["tier_median_ctr"] - eligible["ctr"]

flagged = eligible[eligible["ctr_gap"] > 0].copy()
flagged["score"] = flagged["ctr_gap"] * flagged["impressions_90d"]
flagged["reason_code"] = "position_ok_ctr_below_tier_peers"
flagged["action"] = "review_listing_ctr"

queue_cols = [
    "content_id", "client_id", "position_tier", "avg_position",
    "ctr", "tier_median_ctr", "ctr_gap", "impressions_90d",
    "freshness_tier", "main_intent", "content_type",
    "score", "reason_code", "action",
]
queue = flagged[queue_cols].sort_values("score", ascending=False).reset_index(drop=True)

print(f"Eligible (well-ranked, enough volume): {len(eligible):,}")
print(f"Flagged (below their own tier median): {len(flagged):,} ({len(flagged) / len(eligible):.0%} of eligible)")

os.makedirs("../outputs", exist_ok=True)
out_path = "../outputs/baseline_action_score.csv"
queue.to_csv(out_path, index=False)
print(f"Wrote {len(queue):,} rows to {out_path}")
queue.head(10)


Eligible (well-ranked, enough volume): 15,069
Flagged (below their own tier median): 7,393 (49% of eligible)
Wrote 7,393 rows to ../outputs/baseline_action_score.csv


,content_id,client_id,position_tier,avg_position,ctr,tier_median_ctr,ctr_gap,impressions_90d,freshness_tier,main_intent,content_type,score,reason_code,action
0,content_36ff89c8214e,client_19581e27de,page_1,7.3,0.05,0.23,0.18,295097,91-180,informational,keyword article,53117.46,position_ok_ctr_below_tier_peers,review_listing_ctr
1,content_c8e9d6ab9013,client_19581e27de,page_1,9.7,0.00,0.23,0.23,208678,91-180,informational,keyword article,47995.94,position_ok_ctr_below_tier_peers,review_listing_ctr
2,content_5fe46e04994d,client_4e07408562,page_1,4.2,0.14,0.23,0.09,517715,91-180,informational,keyword article,46594.35,position_ok_ctr_below_tier_peers,review_listing_ctr
3,content_c84a0ab98e90,client_f369cb89fc,page_1,7.8,0.03,0.23,0.20,223271,0-30,informational,keyword article,44654.20,position_ok_ctr_below_tier_peers,review_listing_ctr
4,content_8451fc6f034d,client_d029fa3a95,top_3,2.3,0.03,0.19,0.16,272144,0-30,informational,keyword article,43543.04,position_ok_ctr_below_tier_peers,review_listing_ctr
5,content_453722754fea,client_f369cb89fc,page_1,7.6,0.01,0.23,0.22,140079,0-30,informational,keyword article,30817.38,position_ok_ctr_below_tier_peers,review_listing_ctr
6,content_73c54f78c06a,client_f369cb89fc,page_1,4.7,0.10,0.23,0.13,213963,0-30,informational,keyword article,27815.19,position_ok_ctr_below_tier_peers,review_listing_ctr
7,content_91652435f57a,client_19581e27de,page_1,7.8,0.06,0.23,0.17,159590,91-180,commercial,keyword article,27130.30,position_ok_ctr_below_tier_peers,review_listing_ctr
8,content_c1fe78bc4e37,client_19581e27de,page_1,7.5,0.03,0.23,0.20,134055,91-180,commercial,keyword article,26811.00,position_ok_ctr_below_tier_peers,review_listing_ctr
9,content_0919dd345d80,client_4e07408562,page_1,7.0,0.02,0.23,0.21,119217,0-30,informational,keyword article,25035.57,position_ok_ctr_below_tier_peers,review_listing_ctr


## 3. Top-10 review

Reading the actual top 10 by hand -- action, why it's there, and what would make each one wrong.


In [3]:
top10 = queue.head(10).copy()

for i, row in top10.iterrows():
    print(f"#{i+1}  content={row['content_id']}  client={row['client_id']}")
    print(f"    action: {row['action']}   reason: {row['reason_code']}")
    print(f"    why: {row['position_tier']} tier (median CTR {row['tier_median_ctr']:.2f}) but this "
          f"page's CTR is {row['ctr']:.2f} across {int(row['impressions_90d']):,} impressions "
          f"-> gap {row['ctr_gap']:.2f}, score {row['score']:.0f}")
    if row["ctr"] == 0:
        wrong_if = (f"this is an exact-zero CTR on {int(row['impressions_90d']):,} impressions -- "
                    "that smells like a tracking/indexing artifact (e.g. redirect, canonical issue, "
                    "GSC gap) rather than a genuine listing problem; verify the page actually resolves "
                    "before assigning an editor to rewrite its title")
    else:
        wrong_if = ("the gap reflects a real intent/SERP-feature ceiling (a featured snippet or PAA box "
                    "eating clicks above this result) rather than a fixable title/meta problem")
    print(f"    would be WRONG if: {wrong_if}")
    print()


#1  content=content_36ff89c8214e  client=client_19581e27de
    action: review_listing_ctr   reason: position_ok_ctr_below_tier_peers
    why: page_1 tier (median CTR 0.23) but this page's CTR is 0.05 across 295,097 impressions -> gap 0.18, score 53117
    would be WRONG if: the gap reflects a real intent/SERP-feature ceiling (a featured snippet or PAA box eating clicks above this result) rather than a fixable title/meta problem

#2  content=content_c8e9d6ab9013  client=client_19581e27de
    action: review_listing_ctr   reason: position_ok_ctr_below_tier_peers
    why: page_1 tier (median CTR 0.23) but this page's CTR is 0.00 across 208,678 impressions -> gap 0.23, score 47996
    would be WRONG if: this is an exact-zero CTR on 208,678 impressions -- that smells like a tracking/indexing artifact (e.g. redirect, canonical issue, GSC gap) rather than a genuine listing problem; verify the page actually resolves before assigning an editor to rewrite its title

#3  content=content_5fe46e0499

## 4. Weak picks + leakage check

**Weak pick, found by actually reading the top 10 (not assumed in advance):** row #2 has an
exact **0.00 CTR on 208,678 impressions**. Zero clicks across that much visibility is unusual
enough to be a red flag for a data/tracking issue (broken canonical, redirect, or a GSC
reporting gap) rather than genuine "nobody wants to click this." An editor should confirm the
page actually resolves and is indexed correctly before spending time on a title rewrite. I went
in expecting the weak pick to be an intent mismatch (navigational/branded) -- the top 10 turned
out to be entirely `informational`/`commercial`, so that specific worry didn't apply; the real
weak spot was one I only found by reading the rows, not by guessing beforehand.

**Leakage check:** the score uses only `ctr`, `avg_position`/`position_tier`, and
`impressions_90d` -- all current-snapshot observed columns, none derived from a future window.
Explicitly NOT used: `impressions_last_30d` / `impressions_prev_30d` / `trend_direction` /
`trend_pct` (forward/trend-looking columns belonging to the *other* lane's label), and no
FlyRank product flags (`health_score`, `priority_score`, `action_type`) -- those were never
shipped in this dataset, so there was nothing to accidentally leak in.


In [4]:
# Concrete weak-pick check: any exact-zero-CTR rows in the top 10 despite big traffic?
weak = top10[top10["ctr"] == 0]
print(f"Top-10 rows with exact-zero CTR on real traffic (possible tracking artifact): {len(weak)}")
if len(weak):
    print(weak[["content_id", "ctr", "impressions_90d", "tier_median_ctr", "score"]])

# Leakage check: confirm none of the excluded columns fed the score
used_cols = {"ctr", "avg_position", "position_tier", "impressions_90d"}
excluded_cols = {"impressions_last_30d", "impressions_prev_30d", "trend_direction", "trend_pct",
                  "health_score", "priority_score", "action_type"}
print()
print("Score inputs:", sorted(used_cols))
print("Confirmed NOT used:", sorted(excluded_cols & set(df.columns)), "(present in data but excluded from score)")


Top-10 rows with exact-zero CTR on real traffic (possible tracking artifact): 1
             content_id  ctr  impressions_90d  tier_median_ctr     score
1  content_c8e9d6ab9013  0.0           208678             0.23  47995.94

Score inputs: ['avg_position', 'ctr', 'impressions_90d', 'position_tier']
Confirmed NOT used: ['impressions_last_30d', 'impressions_prev_30d', 'trend_direction', 'trend_pct'] (present in data but excluded from score)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.